# Level-Adaptive IELTS Feedback

Scores an essay with the fine-tuned DeBERTa model from `Fine_tuning_Essay_Prediction_cleaned.ipynb`,
then asks Gemini to explain that score against the official band descriptors.

**Who does what**

* **DeBERTa decides the band.** Reproducible, and measured on 878 held-out essays (QWK 0.605).
* **Gemini only explains.** The predicted band is given to it in the prompt and it is told not to
  re-score, so the feedback can never contradict the number the student sees.

In [32]:
!pip install -q google-genai transformers sentencepiece

In [33]:
import os, sys, json, getpass
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

MODEL_DIR = "/content/drive/MyDrive/ielts-band-coral"
sys.path.append(MODEL_DIR)       

print(sorted(os.listdir(MODEL_DIR)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['__pycache__', 'coral_cutoffs.npy', 'head_config.json', 'model.py', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


## 1. Load the scorer

In [34]:
from transformers import AutoTokenizer
from safetensors.torch import load_file
from model import Deberta           # the class file saved by the training notebook

config  = json.load(open(f"{MODEL_DIR}/head_config.json"))
cutoffs = np.load(f"{MODEL_DIR}/coral_cutoffs.npy")

# JSON keys are always strings, so turn them back into ints
id_to_band = {int(k): v for k, v in config["id_to_band"].items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

scorer = Deberta(
    config["model_name"],
    num_classes=config["num_classes"],
    dropout_rate=config["dropout_rate"],
    n_dropout_samples=config["n_dropout_samples"],
)
scorer.load_state_dict(load_file(f"{MODEL_DIR}/model.safetensors"))
scorer.eval()

print("bands:", id_to_band)
print("cutoffs:", cutoffs.round(2))

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bands: {0: 4.0, 1: 5.0, 2: 6.0, 3: 7.0, 4: 8.0}
cutoffs: [0.49 0.53 0.43 0.45]


In [35]:
def score_essay(topic, essay):
    """Predict a band. No padding: a single essay never needs it."""
    text = f"{topic} {tokenizer.sep_token} {essay}"
    enc  = tokenizer(text, return_tensors="pt", truncation=True,
                     max_length=config["max_length"])

    with torch.no_grad():
        logits = scorer(input_ids=enc["input_ids"],
                        attention_mask=enc["attention_mask"])["logits"][0]

    probs = torch.sigmoid(logits).numpy()
    return id_to_band[int((probs > cutoffs).sum())]


print("test:", score_essay("Do you agree?", "This is a very short and weak answer."))

test: 4.0


## 2. The rubric

Official IELTS Writing Task 2 band descriptors, public version, updated May 2023.

Only bands 4–9 are included: the scorer floors everything at 4.0, and 9.0 is as high as
"the band above" can ever reach. Sentences that the source shows in bold are negative
features that limit a rating — they are kept here as plain text.

In [36]:
CRITERIA = [
    "Task Response",
    "Coherence & Cohesion",
    "Lexical Resource",
    "Grammatical Range & Accuracy",
]

RUBRIC = {
    9.0: {
        "Task Response":
            "The prompt is appropriately addressed and explored in depth. A clear and fully "
            "developed position is presented which directly answers the question/s. Ideas are "
            "relevant, fully extended and well supported. Any lapses in content or support are "
            "extremely rare.",
        "Coherence & Cohesion":
            "The message can be followed effortlessly. Cohesion is used in such a way that it "
            "very rarely attracts attention. Any lapses in coherence or cohesion are minimal. "
            "Paragraphing is skilfully managed.",
        "Lexical Resource":
            "Full flexibility and precise use are widely evident. A wide range of vocabulary is "
            "used accurately and appropriately with very natural and sophisticated control of "
            "lexical features. Minor errors in spelling and word formation are extremely rare "
            "and have minimal impact on communication.",
        "Grammatical Range & Accuracy":
            "A wide range of structures is used with full flexibility and control. Punctuation "
            "and grammar are used appropriately throughout. Minor errors are extremely rare and "
            "have minimal impact on communication.",
    },
    8.0: {
        "Task Response":
            "The prompt is appropriately and sufficiently addressed. A clear and well-developed "
            "position is presented in response to the question/s. Ideas are relevant, well "
            "extended and supported. There may be occasional omissions or lapses in content.",
        "Coherence & Cohesion":
            "The message can be followed with ease. Information and ideas are logically "
            "sequenced, and cohesion is well managed. Occasional lapses in coherence and "
            "cohesion may occur. Paragraphing is used sufficiently and appropriately.",
        "Lexical Resource":
            "A wide resource is fluently and flexibly used to convey precise meanings. There is "
            "skilful use of uncommon and/or idiomatic items when appropriate, despite occasional "
            "inaccuracies in word choice and collocation. Occasional errors in spelling and/or "
            "word formation may occur, but have minimal impact on communication.",
        "Grammatical Range & Accuracy":
            "A wide range of structures is flexibly and accurately used. The majority of "
            "sentences are error-free, and punctuation is well managed. Occasional, "
            "non-systematic errors and inappropriacies occur, but have minimal impact on "
            "communication.",
    },
    7.0: {
        "Task Response":
            "The main parts of the prompt are appropriately addressed. A clear and developed "
            "position is presented. Main ideas are extended and supported but there may be a "
            "tendency to over-generalise or there may be a lack of focus and precision in "
            "supporting ideas/material.",
        "Coherence & Cohesion":
            "Information and ideas are logically organised, and there is a clear progression "
            "throughout the response. (A few lapses may occur, but these are minor.) A range of "
            "cohesive devices including reference and substitution is used flexibly but with "
            "some inaccuracies or some over/under use. Paragraphing is generally used "
            "effectively to support overall coherence, and the sequencing of ideas within a "
            "paragraph is generally logical.",
        "Lexical Resource":
            "The resource is sufficient to allow some flexibility and precision. There is some "
            "ability to use less common and/or idiomatic items. An awareness of style and "
            "collocation is evident, though inappropriacies occur. There are only a few errors "
            "in spelling and/or word formation and they do not detract from overall clarity.",
        "Grammatical Range & Accuracy":
            "A variety of complex structures is used with some flexibility and accuracy. Grammar "
            "and punctuation are generally well controlled, and error-free sentences are "
            "frequent. A few errors in grammar may persist, but these do not impede "
            "communication.",
    },
    6.0: {
        "Task Response":
            "The main parts of the prompt are addressed (though some may be more fully covered "
            "than others). An appropriate format is used. A position is presented that is "
            "directly relevant to the prompt, although the conclusions drawn may be unclear, "
            "unjustified or repetitive. Main ideas are relevant, but some may be insufficiently "
            "developed or may lack clarity, while some supporting arguments and evidence may be "
            "less relevant or inadequate.",
        "Coherence & Cohesion":
            "Information and ideas are generally arranged coherently and there is a clear "
            "overall progression. Cohesive devices are used to some good effect but cohesion "
            "within and/or between sentences may be faulty or mechanical due to misuse, overuse "
            "or omission. The use of reference and substitution may lack flexibility or clarity "
            "and result in some repetition or error. Paragraphing may not always be logical "
            "and/or the central topic may not always be clear.",
        "Lexical Resource":
            "The resource is generally adequate and appropriate for the task. The meaning is "
            "generally clear in spite of a rather restricted range or a lack of precision in "
            "word choice. If the writer is a risk-taker, there will be a wider range of "
            "vocabulary used but higher degrees of inaccuracy or inappropriacy. There are some "
            "errors in spelling and/or word formation, but these do not impede communication.",
        "Grammatical Range & Accuracy":
            "A mix of simple and complex sentence forms is used but flexibility is limited. "
            "Examples of more complex structures are not marked by the same level of accuracy as "
            "in simple structures. Errors in grammar and punctuation occur, but rarely impede "
            "communication.",
    },
    5.0: {
        "Task Response":
            "The main parts of the prompt are incompletely addressed. The format may be "
            "inappropriate in places. The writer expresses a position, but the development is "
            "not always clear. Some main ideas are put forward, but they are limited and are not "
            "sufficiently developed and/or there may be irrelevant detail. There may be some "
            "repetition.",
        "Coherence & Cohesion":
            "Organisation is evident but is not wholly logical and there may be a lack of "
            "overall progression. Nevertheless, there is a sense of underlying coherence to the "
            "response. The relationship of ideas can be followed but the sentences are not "
            "fluently linked to each other. There may be limited/overuse of cohesive devices "
            "with some inaccuracy. The writing may be repetitive due to inadequate and/or "
            "inaccurate use of reference and substitution. Paragraphing may be inadequate or "
            "missing.",
        "Lexical Resource":
            "The resource is limited but minimally adequate for the task. Simple vocabulary may "
            "be used accurately but the range does not permit much variation in expression. "
            "There may be frequent lapses in the appropriacy of word choice and a lack of "
            "flexibility is apparent in frequent simplifications and/or repetitions. Errors in "
            "spelling and/or word formation may be noticeable and may cause some difficulty for "
            "the reader.",
        "Grammatical Range & Accuracy":
            "The range of structures is limited and rather repetitive. Although complex "
            "sentences are attempted, they tend to be faulty, and the greatest accuracy is "
            "achieved on simple sentences. Grammatical errors may be frequent and cause some "
            "difficulty for the reader. Punctuation may be faulty.",
    },
    4.0: {
        "Task Response":
            "The prompt is tackled in a minimal way, or the answer is tangential, possibly due "
            "to some misunderstanding of the prompt. The format may be inappropriate. A position "
            "is discernible, but the reader has to read carefully to find it. Main ideas are "
            "difficult to identify and such ideas that are identifiable may lack relevance, "
            "clarity and/or support. Large parts of the response may be repetitive.",
        "Coherence & Cohesion":
            "Information and ideas are evident but not arranged coherently and there is no clear "
            "progression within the response. Relationships between ideas can be unclear and/or "
            "inadequately marked. There is some use of basic cohesive devices, which may be "
            "inaccurate or repetitive. There is inaccurate use or a lack of substitution or "
            "referencing. There may be no paragraphing and/or no clear main topic within "
            "paragraphs.",
        "Lexical Resource":
            "The resource is limited and inadequate for or unrelated to the task. Vocabulary is "
            "basic and may be used repetitively. There may be inappropriate use of lexical "
            "chunks (e.g. memorised phrases, formulaic language and/or language from the input "
            "material). Inappropriate word choice and/or errors in word formation and/or in "
            "spelling may impede meaning.",
        "Grammatical Range & Accuracy":
            "A very limited range of structures is used. Subordinate clauses are rare and simple "
            "sentences predominate. Some structures are produced accurately but grammatical "
            "errors are frequent and may impede meaning. Punctuation is often faulty or "
            "inadequate.",
    },
}

print("bands:", sorted(RUBRIC))
print("criteria per band:", {b: len(d) for b, d in RUBRIC.items()})

bands: [4.0, 5.0, 6.0, 7.0, 8.0, 9.0]
criteria per band: {9.0: 4, 8.0: 4, 7.0: 4, 6.0: 4, 5.0: 4, 4.0: 4}


## 3. Cambridge exemplars (optional)

A CSV on Drive with columns `topic`, `essay`, `band`. Used to show the model one real
answer at the band the student is aiming for. The notebook works without it.

In [37]:
# Cambridge data produced by Preprocessing_Raw_Datasets.ipynb
# Schema: topic, band, essay, examiner_comment, source
EXEMPLAR_PATH = "/content/drive/MyDrive/cambridge_trocr_gector.jsonl"

exemplars = pd.read_json(EXEMPLAR_PATH, lines=True)
print(f"{len(exemplars)} rows | bands:", sorted(exemplars["band"].dropna().unique()))

# Show a real answer at the target band. The text came from handwriting OCR
# and is still rough, so the chosen exemplar is printed below - check it
# reads like English before trusting the feedback that references it.
USE_ESSAY_EXEMPLARS = True

# The examiner comments are printed text, OCR'd far more reliably, and make a
# good tone model. Skip any that name a band, or the model starts re-scoring.
comments = exemplars["examiner_comment"].fillna("").str.strip()
usable = comments[(comments != "") & (~comments.str.contains(r"[Bb]and\s*\d"))]

# Fallback: two comments from the Kaggle file (a coaching site, not Cambridge)
KAGGLE_STYLE = [
    "Where are the paragraphs in this essay? You must be very careful using "
    "definitive words such as 'always' and making statements about facts. The "
    "essay is for you to provide an opinion and to provide supporting arguments.",

    "This is a very good essay, you have made your arguments well and set out the "
    "paragraphs as required. However, pay attention to your use of assertive "
    "statements e.g. 'Without cigarettes, these people would have no jobs'. "
    "Perhaps they would gain employment in another industry - we cannot be sure.",
]

STYLE_EXAMPLES = usable.head(2).tolist() or KAGGLE_STYLE

print("style source:", "Cambridge" if len(usable) else "Kaggle fallback")
for s in STYLE_EXAMPLES:                 # read these - OCR quality varies
    print("  -", s[:110])


14 rows | bands: [np.float64(4.5), np.float64(6.0), np.float64(6.5), np.float64(7.0)]
style source: Cambridge
  - The candidate clearly explains why home ownership may be of importance to some people she or he also explores 
  - This is a thoughtful exploration of the topic the writer considers the advantages of having online materials r


## 4. Build the prompt

The level-adaptive part: the model is shown the descriptors for the student's band **and
the band above**, so the feedback becomes the gap between the two.

In [38]:
FEEDBACK_LANGUAGE = "Korean"    

# ONE complete worked example does more for quality than any model upgrade.
# Write the feedback you actually want for a single essay and paste it here.
# Leave as "" until you have one - a bad example is worse than none.
WORKED_EXAMPLE = ""


def build_prompt(topic, essay, band):
    next_band = min(band + 1.0, max(RUBRIC))

    def descriptors(b):
        return "\n".join(f"- {c}: {RUBRIC[b][c]}" for c in CRITERIA)

    example = ""
    if USE_ESSAY_EXEMPLARS:
        match = exemplars[exemplars["band"] == next_band]
        if len(match):
            example = f"\nFor reference, a real band {next_band} answer:\n{match.iloc[0]['essay']}\n"

    worked = f"\nThis is the standard to match:\n{WORKED_EXAMPLE}\n" if WORKED_EXAMPLE else ""

    criteria_list = "\n".join(f"- {c}" for c in CRITERIA)
    style = "\n\n".join(STYLE_EXAMPLES)

    return f"""Question: {topic}

Student's essay:
{essay}

A separate scoring model has assessed this essay as band {band}.
Treat that band as fixed and correct. Do not re-score the essay.

Band {band} descriptors - where the student is now:
{descriptors(band)}

Band {next_band} descriptors - where the student is going:
{descriptors(next_band)}
{example}
Write one short paragraph per criterion, in this order:
{criteria_list}

Every paragraph must:
- quote one exact phrase from THIS essay, in quotation marks
- say what specifically about that phrase keeps it at band {band}
- rewrite that phrase as it would appear at band {next_band}

Reject any sentence you could write about a different essay without changing it.
If a criterion is already close to band {next_band}, say so briefly rather than
inventing a fault.

Name the single criterion holding this essay back the most, and give that one
noticeably more detail than the other three.

Formatting rules:
- Write in {FEEDBACK_LANGUAGE}.
- Open with two or three sentences: one genuinely earned piece of encouragement,
  then the overall strengths and weaknesses in brief.
- Then exactly four paragraphs, one per criterion, each under its own heading,
  beginning with "{CRITERIA[0]}". No closing paragraph after the fourth.
- Address the student as "you". Under 120 words per paragraph.
{worked}
Match the tone of these real examiner comments - their directness and their habit
of quoting the essay, not their wording, and never their scoring:

{style}"""


print(build_prompt("Sample question", "Sample essay text.", 6.0)[-1000:])


 disappear although other printed materials such as magazines and newspapers may become completely digital sed the score might be improved by further exploration of whether online materials will be free as cost is mentioned only briefly organisation is clear par graphing is logical and linking words and phrases guide the reader through the script a of these factors on the other hand based on this the range of vocabulary is quite varied with many examples of co location electronic devices digital book and newspapers environmentally friendly digital versions traditional printed books emotional connection and value with only two spelling errors reach reach resources resources there is a mix of simple and complex sentence structures and these are generally accurate some errors do occur todays today's fo read book s makes some people to believe makes some people believe digital versions of books is are more convenient printing so such a huge amount of articles but the meaning is still clear

## 5. Generate the feedback

In [39]:
from google import genai
from google.genai import types

# Colab secrets can't be read through the VS Code bridge, so the key is typed in.
api_key = getpass.getpass("Gemini API key: ")
client  = genai.Client(api_key=api_key)

GEMINI_MODEL = "gemini-3.6-flash"   

SYSTEM = (
    "You are an experienced IELTS Writing Task 2 examiner giving feedback to a learner. "
    "You explain a band that has already been decided elsewhere; you never change it. "
    "Your comments on individual criteria are your own reading of the essay."
    "Before you start, read the rubric for the band that has been assigned to the essay, and the rubric for the next band. "
    "Before giving the specific feedback in each section, give a brief summary of the overall strengths and weaknesses of the essay, and first sentence of the feedback needs to be encouraging words"
)


def generate_feedback(topic, essay):
    band = score_essay(topic, essay)

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=build_prompt(topic, essay, band),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM,
            temperature=0.3,        # low, so the same essay gets stable feedback
        ),
    )
    return band, response.text

## 6. Revise and re-score

Gemini rewrites the essay applying its own feedback, then the **same DeBERTa
model** scores the rewrite. The second band is measured, not asserted - Gemini
is never asked what the rewrite is worth.

Two things to be honest about:

* The revision is Gemini's writing, not the student's. Present it as *"a revision
  applying this feedback scores X"*, never as *"your essay would score X"*.
* Using the scorer to validate feedback aimed at pleasing that same scorer is
  circular. It is a useful signal, not proof - and with QWK 0.60 a one-band move
  is within the model's own noise.


In [40]:
def revise_and_rescore(topic, essay, feedback):
    """Apply the feedback, then score the result with DeBERTa - not with Gemini."""
    prompt = f"""Rewrite the essay below so that it acts on every point of the feedback.

Keep the student's own arguments, examples and opinion. It is important to do minimal change in writing, and do not replace the content. Return only the revised essay: no commentary,
no headings, no explanation of what you changed.

Question: {topic}

Essay:
{essay}

Feedback:
{feedback}"""

    revised = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3),
    ).text.strip()

    return revised, score_essay(topic, revised)   # the band comes from the model


In [41]:
topic = 'Nowadays both men and women spend a lot of money on beauty care. This was not so in the past. What may be the root cause of this behaviour? Discuss the reasons and possible results.'

essay = 'In modern society, there is a growing debate over beauty care, with both men and women spending a considerable amount of money on it. There are several reasons for this phenomenon, such as the influence of social media and evolving societal standards, and the effects on people\'s beauty. This essay will delve into the reasons for this undesirable trend and its consequences. One of the primary reasons for the increased spending on beauty care is the influence of social media. Platforms like Instagram, TikTok, and Youtube are filled with influencers and celebrities who set high beauty standards. These individuals often promote various beauty products, encouraging their followers to emulate their looks. Therefore, the constant exposure of influencers on social media leads to spending on beauty products and services. Another contributing factor is the evolution of societal standards. In modern society, looking good can positively impact various aspects of an individual\'s life, including their professional and social spheres. For example, a well-groomed appearance can be well perceived by employers. A primary outcome of increased spending on beauty is the growth of the beauty industry. This industry creates numerous job opportunities and contributes to the economy. However, there are also negative consequences associated with this trend. The pressure to conform to idealized beauty standards can lead to mental health issues. Constant exposure to flawless images on social media can result in self-dissatisfaction. This can cause depression and low self-esteem, particularly among young people. To sum up, the growing expenditure on beauty care in modern society has both positive and negative results. While it contributes to economic growth, it also poses risks to mental health.'

band, feedback = generate_feedback(topic, essay)
revised, revised_band = revise_and_rescore(topic, essay, feedback)

print(f"PREDICTED BAND: {band}   ({len(essay.split())} words)\n")
print(feedback)

print("\n" + "=" * 70)
print(f"REVISED VERSION - scored {revised_band} by the same model, "
      f"{len(revised.split())} words")
print("=" * 70 + "\n")
print(revised)


PREDICTED BAND: 6.0   (272 words)

작성자님의 에세이는 질문의 핵심 주제를 정확히 파악하고 전체적으로 다듬어진 문장으로 논리를 전달하고 있어 매우 훌륭합니다. 이유와 결과를 균형 있게 배치한 점은 좋으나, 서론과 본문 간의 관점 불일치와 일부 아이디어의 발전 부족이 아쉬운 요소입니다.

### Task Response
"This essay will delve into the reasons for this undesirable trend and its consequences."
이 구문은 작성자님의 점수를 6.0에 머물게 한 가장 결정적인 요인입니다. 서론에서는 이 현상을 단순 'undesirable trend(바람직하지 않은 경향)'로 규정했으나, 정작 본문에서는 아름다움에 대한 지출이 경제와 고용에 미치는 긍정적 영향도 함께 다루어 전체적인 포지션의 일관성이 흔들립니다. 또한 질문에서 강조한 '과거와 달리 왜 남녀 모두가 지출하는지'에 대한 시대적/성별적 원인 분석을 충분히 확장하지 못하고 일반적인 원인만 나열했습니다.
Band 7.0 수정안: "This essay will examine the primary drivers behind this shifting behavior and evaluate both its economic impacts and psychological consequences."

### Coherence & Cohesion
"Therefore, the constant exposure of influencers on social media leads to spending on beauty products and services."
이 문장에서는 접속사 'Therefore'가 다소 기계적으로 사용되었으며, 인플루언서에 대한 '노출'이 곧바로 개인의 소비로 이어진다는 문장 간 논리적 연결이 다소 비약되어 Band 6.0에 머물렀습니다.
Band 7.0 수정안: "Consequently, continuous exp